In [ ]:
import time

from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.1.253")
got.show_light_rgb([1, 2, 3], 255, 255, 255)

got.load_models(['apriltag_qrcode'])

In [ ]:
print(got.get_apriltag_total_info())

## Recognize AprilTag's ID

In [ ]:
target_id = 3

while True:
    got.mecanum_turn_speed(3, 20)
    info = got.get_apriltag_total_info()
    
    if info:
        aptag_id, center_x, center_y, height, width, area, distance5x5, *_ = info[0]
        
        if aptag_id == target_id:
            break

got.mecanum_stop()

In [ ]:
target_id = 3

target_x = 320
target_width = 150

x_tolerance = 50
width_tolerance = 10

max_x_speed = 15
max_y_speed = 15

def get_aptag_info():
    info = got.get_apriltag_total_info()

    if info:
        aptag_id, center_x, _, _, width, *_ = info[0]

        if aptag_id == target_id:
            return center_x, width

    return None

def align_and_approach():
    while True:
        info = get_aptag_info()

        if info is None:
            got.mecanum_stop()
            return False

        center_x, width = info

        # Calculate errors
        x_error = center_x - target_x
        width_error = target_width - width

        # Horizontal movement
        if abs(x_error) > x_tolerance:
            x_speed = int(max(-max_x_speed, min(max_x_speed, x_error * 0.2)))
        else:
            x_speed = 0

        # Forward movement
        if width_error > width_tolerance:
            y_speed = int(max(0, min(max_y_speed, width_error * 0.5)))
        else:
            y_speed = 0

        # Both conditions satisfied
        if x_speed == 0 and y_speed == 0:
            got.mecanum_stop()
            return True

        # Move horizontally and forward simultaneously
        got.mecanum_move_xyz(x_speed, y_speed, 0)

        print(f"X Error: {x_error}, Width Error: {width_error}")
        print(f"X Speed: {x_speed}, Y Speed: {y_speed}")

        time.sleep(0.1)

def pickup():
    info = get_aptag_info()
    
    if info is None:
        return False
    
    center_x, _ = info

    got.mechanical_clamp_release()
    got.mechanical_joint_control(int(320 - center_x) * 0.1, -40, 0, 500)
    time.sleep(1)
    got.mechanical_clamp_close()
    time.sleep(1)
    got.mechanical_joint_control(90, 45, 45, 500)
    time.sleep(1)

try:
    while True:
        info = get_aptag_info()

        if info is None:
            got.mecanum_turn_speed(3, 20)
            time.sleep(0.1)
            continue

        got.mecanum_stop()

        if align_and_approach():
            pickup()
            break
finally:
    got.mecanum_stop()